In [4]:
import torch
from transformers import AutoTokenizer, EsmModel
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [5]:
# ==========================================
# 1. SETUP HARDWARE & LOAD ESM-2
# ==========================================
# Use GPU if available (makes embedding extraction 50x faster)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the 150-million parameter ESM-2 model and its tokenizer
model_name = "facebook/esm2_t30_150M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)
model.eval()  # Put the model in evaluation mode

Using device: cpu


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  595MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream tas

EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 640, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-29): 30 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=640, out_features=640, bias=True)
            (key): Linear(in_features=640, out_features=640, bias=True)
            (value): Linear(in_features=640, out_features=640, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=640, out_features=640, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((640,), eps=1e-05, elementwise_affine=True, bias=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=640, out_features=2560, bias=True)
        )
        (output): EsmOutput(
  

In [6]:
# ==========================================
# 2. DEFINE EMBEDDING EXTRACTION FUNCTION
# ==========================================
def get_mean_embedding(sequence):
    """
    Takes a single amino acid sequence, passes it through ESM-2,
    and returns a single 1D vector (length 640 for the 150M model)
    representing the average properties of that protein.
    """
    # Tokenize the input sequence and move to GPU/CPU
    inputs = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract the last hidden states (shape: [1, sequence_length, embedding_dimension])
    last_hidden_states = outputs.last_hidden_state
    
    # Calculate the mean across the sequence length dimension (collapsing it to 1D)
    mean_embedding = torch.mean(last_hidden_states, dim=1).squeeze()
    
    # Move back to CPU memory and convert to a standard NumPy array
    return mean_embedding.cpu().numpy()

Explanation:
* `Transformer`: a Python library developped by *Hgging Face*. It's a translaotr for deep learning models.
* `AutoTokenizer`: the function from `transformer` use to make tokens from sequences.
* PyTorch (pt) tensor: data structure in the PyTorch deep learning framework. Like a multi-dimensional array. Capbility to run on GPUs.
* `max_length`: the max legnth of aa sequences.
* `torch.no_grad()`: turn off the **graident** here when not training model, for example,extracting embeddings or making prediction using the model.
* Gradient: a vector that points the direction of a function. The **gradient of the loss function** with respect to the model's adjustable parameters (weights and biases) is particualry focused in deep learning. **Backpropagation** (an algorithm) is used to calculate the gradients, computing how much each parameter contributes to the total error. Then, an **optimizer** uses the gradients to update the parameters. The iterative process of calculating gradients and updating parameters is how a neural network learns.
* `outputs = model(**inputs)`: `model` = EsmModel (esm2_t30_150M_UR50D). `inputs`: the tokenized sequence **dictionary**. `**` to unpack a dictionary and treat them as separate keyword arguments (e.g., `{'input_ids': tensor_a, 'attention_mask': tensor_b}` --> `model(**input)` = `model(input_ids=tensor_a, attention_mask=tensor_b)`.

In [7]:
# ==========================================
# 3. MOCK DATA SETUP (Replace this with your real CSV load)
# ==========================================
# Let's assume you have a pandas DataFrame with your sequences and labels.
# Target label: 0 for 18:0-preferring (stearoyl), 1 for 16:0-preferring (palmitoyl)
print("\nCreating mock dataset...")
data = {
    "sequence": ["MSDNG" * 20, "MADSG" * 20, "MVKES" * 20, "MGGKA" * 20] * 250, # 1,000 mock sequences
    "specificity": [0, 1, np.nan, np.nan] * 250 # 500 labeled (0/1), 500 unlabeled (NaN)
}
df = pd.DataFrame(data)
print(df)


Creating mock dataset...
                                              sequence  specificity
0    MSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGM...          0.0
1    MADSGMADSGMADSGMADSGMADSGMADSGMADSGMADSGMADSGM...          1.0
2    MVKESMVKESMVKESMVKESMVKESMVKESMVKESMVKESMVKESM...          NaN
3    MGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAM...          NaN
4    MSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGM...          0.0
..                                                 ...          ...
995  MGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAM...          NaN
996  MSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGMSDNGM...          0.0
997  MADSGMADSGMADSGMADSGMADSGMADSGMADSGMADSGMADSGM...          1.0
998  MVKESMVKESMVKESMVKESMVKESMVKESMVKESMVKESMVKESM...          NaN
999  MGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAMGGKAM...          NaN

[1000 rows x 2 columns]


In [8]:
# ==========================================
# 4. GENERATE EMBEDDINGS FOR ALL SEQUENCES
# ==========================================
print("Extracting ESM-2 embeddings (This may take a minute)...")
embeddings_list = []
for seq in df["sequence"]:
    emb = get_mean_embedding(seq)
    embeddings_list.append(emb)

# Convert the list of embeddings into a clean 2D NumPy array (shape: 1000, 640)
X_all = np.array(embeddings_list)
print(f"Generated embedding matrix shape: {X_all.shape}")

Extracting ESM-2 embeddings (This may take a minute)...
Generated embedding matrix shape: (1000, 640)


Explanation:
* `seq in df["sequence"]`: loop through each sequence in the **sequence** column of the dataframe created earlier.
* `embeddings_list = []`: save each protein's embedding here.
*  `get_mean_embedding(seq)`: This function takes the raw sequence, tokenizes it, passes it through the ESM-2 model, performs mean pooling, and returns a 640-dimensional NumPy array representing the embedding for that specific protein.

In [9]:
# show the first 5 rows of the resulting embedding
display(X_all[:5])

array([[ 0.10003058, -0.41478595, -0.19941574, ..., -0.27906448,
         0.01984723,  0.01753037],
       [ 0.1132565 , -0.29896453, -0.10906006, ..., -0.3883977 ,
        -0.04256304,  0.0357337 ],
       [ 0.04313174, -0.18065587, -0.0564288 , ..., -0.26919302,
        -0.08039302,  0.14769028],
       [ 0.06270251, -0.47664422, -0.07103497, ..., -0.3428464 ,
        -0.14366536, -0.07485849],
       [ 0.10003058, -0.41478595, -0.19941574, ..., -0.27906448,
         0.01984723,  0.01753037]], shape=(5, 640), dtype=float32)

In [10]:
# ==========================================
# 5. SPLIT LABELED VS. UNLABELED DATA
# ==========================================
# Create a mask to separate the rows that have known specificity labels
labeled_mask = df["specificity"].notna()

# X_train & y_train: The data with known labels
X_train = X_all[labeled_mask]
y_train = df.loc[labeled_mask, "specificity"].astype(int).values

# X_predict: The data we want to make predictions on
X_predict = X_all[~labeled_mask]

print(f"Labeled training samples: {X_train.shape[0]}")
print(f"Unlabeled prediction samples: {X_predict.shape[0]}")

Labeled training samples: 500
Unlabeled prediction samples: 500


Explanation:
* `df["specificity"].notna()`: selects the **specificity** column from my DataFrame and filter for the labeled data by `notna()`.
* `df.loc[labeled_mask, "specificity"]` selects the 'specificity' column values from your DataFrame, again using the labeled_mask to get only the labels for the labeled proteins. `astype(int)` converts to integer, and `.values` extract the NumPy array.
*  `X_all[~labeled_mask]` is basically the opposite to `X_all[labeled_mask]` where it picks the unlabeled data and assign it as dthe data for prediction.

In [11]:
# ===============================================
# 6. SPLIT Into Training/Validation/Test Datasets
# ===============================================

# 1. First, split into (Train + Validation) and Test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Now split the temporary set into final Train and Validation
# Since 0.15 is roughly 17.6% of 0.85, we use that for the second split
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Final Training set: {X_train_final.shape[0]} samples")
print(f"Validation set:     {X_val.shape[0]} samples")
print(f"Test set:           {X_test.shape[0]} samples")

Final Training set: 350 samples
Validation set:     75 samples
Test set:           75 samples


In [12]:
# ==========================================
# 7. TRAIN CLASSIFIER & EVALUATE
# ==========================================

# 1. Initialize and train the classifier on the FINAL training split
# We do this here to ensure 'classifier' is defined in the current session
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train_final, y_train_final)

# 2. Use the model to predict labels for the held-out test set
y_test_pred = classifier.predict(X_test)

# 3. Calculate the overall accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Final Unbiased Test Accuracy: {test_accuracy * 100:.2f}%")

# 4. Print a detailed classification report (Precision, Recall, F1-score)
print("\nClassification Report on Test Set:")
print(classification_report(y_test, y_test_pred, target_names=['18:0-preferring', '16:0-preferring']))

Final Unbiased Test Accuracy: 100.00%

Classification Report on Test Set:
                 precision    recall  f1-score   support

18:0-preferring       1.00      1.00      1.00        38
16:0-preferring       1.00      1.00      1.00        37

       accuracy                           1.00        75
      macro avg       1.00      1.00      1.00        75
   weighted avg       1.00      1.00      1.00        75



### 7.1 Intermediate Validation Check
Before we trust the final test results, we use the **Validation Set** to see how the model is performing. If the accuracy here is much higher than the test set, it might indicate 'overfitting'.

In [13]:
# Predict on the validation set
y_val_pred = classifier.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Set Accuracy: {val_accuracy * 100:.2f}%")

Validation Set Accuracy: 100.00%


### Optional: Robustness Check with Cross-Validation
If you want to see how stable the model is across different subsets of your training data, you can run cross-validation on just the `X_train_final` portion.

In [14]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation on ONLY the training split
cv_scores = cross_val_score(classifier, X_train_final, y_train_final, cv=5)

print(f"Cross-Validation Accuracy (on training split): {cv_scores.mean() * 100:.2f}% (+/- {cv_scores.std() * 100:.2f}%)")

Cross-Validation Accuracy (on training split): 100.00% (+/- 0.00%)


Explanation:
* `n_estimators=100`: number of decision trees the random forest will build. More trees generally lead to better performance but also increase comput
* `random_state=42`: sets the seed for the random number generator, ensuring that your results are reproducible. Running the code with the same seed# produces the same trees.
* `cross_val_score`: a function from **scikit-learn** used for K-fold cross-validation. Takes an untrained model (`classifier`), training features (`X_train`) and training labels (`y_train`). `cv=5` means 5-fold cross-validation. The data (`X_train`, `y_train`) will be split into **5** equal parts (folds). The model will be trained **5 times**: each time, it **trains on 4 folds and evaluates on the remaining 1 fold**. The scores variable will store the accuracy score from each of these 5 evaluation rounds.
* `classifier.fit(X_train, y_train)`: after cross-validation, performs the final training on **all** labeled data.
* **Validation**: A single 'checkpoint' using a dedicated 15% of your data. It's fast and easy to interpret.
* **Cross-Validation (`cv`)**: A more thorough 'stress test' that shuffles the data multiple times to ensure the results aren't just due to a lucky split.

In [15]:
# ==========================================
# 8. PREDICT FOR UNCHARACTERIZED ENZYMES
# ==========================================
# This uses the 'classifier' trained in Section 7 on the 70% split

# Predict categories (0 or 1)
predictions = classifier.predict(X_predict)

# Predict exact probabilities
probabilities = classifier.predict_proba(X_predict)

# Print some results
print("\n--- SAMPLE PREDICTIONS FOR UNLABELED SADs ---")
for i in range(5):
    target_class = "16:0-preferring (Palmitoyl)" if predictions[i] == 1 else "18:0-preferring (Stearoyl)"
    confidence = probabilities[i][predictions[i]] * 100
    print(f"Enzyme #{i+1} | Prediction: {target_class} | Confidence: {confidence:.2f}%")


--- SAMPLE PREDICTIONS FOR UNLABELED SADs ---
Enzyme #1 | Prediction: 18:0-preferring (Stearoyl) | Confidence: 50.00%
Enzyme #2 | Prediction: 16:0-preferring (Palmitoyl) | Confidence: 66.00%
Enzyme #3 | Prediction: 18:0-preferring (Stearoyl) | Confidence: 50.00%
Enzyme #4 | Prediction: 16:0-preferring (Palmitoyl) | Confidence: 66.00%
Enzyme #5 | Prediction: 18:0-preferring (Stearoyl) | Confidence: 50.00%
